In [2]:
import pandas as pd
from pathlib import Path

file_path = Path("/Users/vanshaj/Work/GitHub/Quant_Labs/Projects/Exploratory Market Analysis/src/data/Equity copy/BHARTIARTL_2024-04-01_to_2025-03-31.parquet")
df = pd.read_parquet(file_path)
print(df.columns)
print(df.head())

Index(['('Date', '')', '('Close', 'BHARTIARTL.NS')',
       '('High', 'BHARTIARTL.NS')', '('Low', 'BHARTIARTL.NS')',
       '('Open', 'BHARTIARTL.NS')', '('Volume', 'BHARTIARTL.NS')'],
      dtype='object')
  ('Date', '')  ('Close', 'BHARTIARTL.NS')  ('High', 'BHARTIARTL.NS')  \
0   2024-04-01                 1200.566895                1214.916348   
1   2024-04-02                 1191.592407                1210.823570   
2   2024-04-03                 1208.801758                1223.496362   
3   2024-04-04                 1190.211670                1216.001176   
4   2024-04-05                 1174.580200                1194.649620   

   ('Low', 'BHARTIARTL.NS')  ('Open', 'BHARTIARTL.NS')  \
0               1191.345823                1211.070092   
1               1186.611981                1208.111483   
2               1173.051521                1178.525054   
3               1184.738258                1214.965603   
4               1172.558415                1189.274804   

   ('

In [8]:
import pandas as pd
from pathlib import Path

def clean_columns(df):
    new_cols = []
    for col in df.columns:
        col_clean = col.strip("()").replace("'", "")
        parts = [p.strip() for p in col_clean.split(",")]
        col_final = "_".join([p for p in parts if p])
        new_cols.append(col_final)
    df.columns = new_cols
    return df

def preprocess_data(df):
    df = clean_columns(df)

    possible_date_cols = ['Date', 'date', 'DATE', 'Timestamp', 'timestamp', 'DATETIME']
    date_col = None
    for col in possible_date_cols:
        if col in df.columns:
            date_col = col
            break

    if date_col is None:
        if isinstance(df.index, pd.DatetimeIndex):
            df = df.copy()
            df['Date'] = df.index
            date_col = 'Date'
        else:
            raise ValueError(f"No date column found. Available columns: {df.columns.tolist()}")

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col).sort_index()

    start_date = df.index.min()
    end_date = df.index.max()
    complete_date_range = pd.date_range(start=start_date, end=end_date, freq='D')

    missing_dates = complete_date_range.difference(df.index)
    if len(missing_dates) > 0:
        print(f"Filling {len(missing_dates)} missing dates")
        df = df.reindex(complete_date_range)
        df = df.ffill().bfill()

    df = df.reset_index().rename(columns={'index': 'Date'})
    df = df.fillna(0)

    return df

def process_parquet_files(data_folder):
    parquet_files = list(Path(data_folder).glob("*.parquet"))

    if not parquet_files:
        print("No parquet files found.")
        return

    for file_path in parquet_files:
        print(f"\nProcessing: {file_path.name}")
        try:
            df = pd.read_parquet(file_path)
            df_processed = preprocess_data(df)
            df_processed.to_parquet(file_path, index=False)
            print("✅ File processed and saved.")
        except Exception as e:
            print(f"❌ Error processing {file_path.name}: {e}")

# Usage
data_folder = "/Users/vanshaj/Work/GitHub/Quant_Labs/Projects/Exploratory Market Analysis/src/data/Equity"
process_parquet_files(data_folder)


Processing: BHARTIARTL_2024-04-01_to_2025-03-31.parquet
Filling 114 missing dates
✅ File processed and saved.

Processing: verizon_2024-04-01_to_2025-03-31.parquet
Filling 112 missing dates
✅ File processed and saved.

Processing: IDEA_2024-04-01_to_2025-03-31.parquet
Filling 114 missing dates
✅ File processed and saved.


In [9]:
import pandas as pd
from pathlib import Path

def clean_columns(df):
    new_cols = []
    for col in df.columns:
        col_clean = col.strip("()").replace("'", "")
        parts = [p.strip() for p in col_clean.split(",")]
        col_final = "_".join([p for p in parts if p])
        new_cols.append(col_final)
    df.columns = new_cols
    return df

def preprocess_data(df):
    df = clean_columns(df)

    possible_date_cols = ['Date', 'date', 'DATE', 'Timestamp', 'timestamp', 'DATETIME']
    date_col = None
    for col in possible_date_cols:
        if col in df.columns:
            date_col = col
            break

    if date_col is None:
        if isinstance(df.index, pd.DatetimeIndex):
            df = df.copy()
            df['Date'] = df.index
            date_col = 'Date'
        else:
            raise ValueError(f"No date column found. Available columns: {df.columns.tolist()}")

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col).sort_index()

    start_date = df.index.min()
    end_date = df.index.max()
    complete_date_range = pd.date_range(start=start_date, end=end_date, freq='D')

    missing_dates = complete_date_range.difference(df.index)
    if len(missing_dates) > 0:
        print(f"Filling {len(missing_dates)} missing dates")
        df = df.reindex(complete_date_range)
        df = df.ffill().bfill()

    df = df.reset_index().rename(columns={'index': 'Date'})
    df = df.fillna(0)

    return df

def process_parquet_files(data_folder):
    parquet_files = list(Path(data_folder).glob("*.parquet"))

    if not parquet_files:
        print("No parquet files found.")
        return

    for file_path in parquet_files:
        print(f"\nProcessing: {file_path.name}")
        try:
            df = pd.read_parquet(file_path)
            df_processed = preprocess_data(df)
            df_processed.to_parquet(file_path, index=False)
            print("✅ File processed and saved.")
        except Exception as e:
            print(f"❌ Error processing {file_path.name}: {e}")

# Usage
data_folder = "/Users/vanshaj/Work/GitHub/Quant_Labs/Projects/Exploratory Market Analysis/src/data/Indices"
process_parquet_files(data_folder)


Processing: snp500_2024-04-01_to_2025-03-31.parquet
Filling 112 missing dates
✅ File processed and saved.

Processing: NIFTY BANK_01-04-2024_to_31-03-2025.parquet
Filling 113 missing dates
❌ Error processing NIFTY BANK_01-04-2024_to_31-03-2025.parquet: cannot reindex on an axis with duplicate labels

Processing: NIFTY 50_01-04-2024_to_31-03-2025.parquet
Filling 113 missing dates
❌ Error processing NIFTY 50_01-04-2024_to_31-03-2025.parquet: cannot reindex on an axis with duplicate labels


/var/folders/y5/g1x0tvb16bbf0rjf7219mhmh0000gn/T/ipykernel_2603/4061552323.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
/var/folders/y5/g1x0tvb16bbf0rjf7219mhmh0000gn/T/ipykernel_2603/4061552323.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
